In [ ]:
import pandas as pd
import numpy as np

Entre as usinas com dados consistentes, quais sofreram mais restrições operativas e qual foi o impacto dessas restrições na geração?

### Etapa 4 — Análise do Constrained-off

# Restrição dados de Restrição de Operação por Constrained-off de Usinas Fotovoltaica

*   val_geracao → Valor da Geração, em MWmed  
*   val_geracaolimitada → Valor da Geração Limitada, em MWmed
*   val_disponibilidade → Valor da Disponibilidade, em MWmed
*   val_geracaoreferencia → Valor da Geração de Referência, em MWmed

**MWmed** é uma unidade de potência média no intervalo de medição, não uma energia acumulada



Possíveis motivos da limitação imposta (Constrained-off): restrições na transmissão, controle de tensão ou segurança operativa   

fontes intermitentes (como solar e eólica), cuja geração depende da natureza, e nem sempre pode ser despachada no momento ideal.

In [ ]:
months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
data_restricao = []

for month in months:
  url = f"https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/restricao_coff_fotovoltaica_tm/RESTRICAO_COFF_FOTOVOLTAICA_2025_{month}.csv"
  df = pd.read_csv(url, sep=';')

  df = df[(df['nom_estado']=='SAO PAULO')]

  data_restricao.append(df)

df_restricao = pd.concat(data_restricao, ignore_index=True)

In [ ]:
df_restricao.head()

,id_subsistema,nom_subsistema,id_estado,nom_estado,nom_usina,id_ons,ceg,din_instante,val_geracao,val_geracaolimitada,val_disponibilidade,val_geracaoreferencia,val_geracaoreferenciafinal,cod_razaorestricao,cod_origemrestricao,dsc_restricao
0,SE,SUDESTE/CENTRO-OESTE,SP,SAO PAULO,CONJ. BOA HORA,CJU_SPUFBH,-,2025-01-01 00:00:00,0.0,NaN,30.017,0.0,NaN,NaN,NaN,NaN
1,SE,SUDESTE/CENTRO-OESTE,SP,SAO PAULO,CONJ. BOA HORA,CJU_SPUFBH,-,2025-01-01 00:30:00,0.0,NaN,30.017,0.0,NaN,NaN,NaN,NaN
2,SE,SUDESTE/CENTRO-OESTE,SP,SAO PAULO,CONJ. BOA HORA,CJU_SPUFBH,-,2025-01-01 01:00:00,0.0,NaN,30.082,0.0,NaN,NaN,NaN,NaN
3,SE,SUDESTE/CENTRO-OESTE,SP,SAO PAULO,CONJ. BOA HORA,CJU_SPUFBH,-,2025-01-01 01:30:00,0.0,NaN,30.082,0.0,NaN,NaN,NaN,NaN
4,SE,SUDESTE/CENTRO-OESTE,SP,SAO PAULO,CONJ. BOA HORA,CJU_SPUFBH,-,2025-01-01 02:00:00,0.0,NaN,30.082,0.0,NaN,NaN,NaN,NaN


In [ ]:
df_restricao['nom_usina'].unique()

array(['CONJ. BOA HORA', 'CONJ. CASTILHO', 'CONJ. DRACENA',
       'CONJ. GUAIMBÊ', 'CONJ. PEDRANÓPOLIS', 'CONJ. PEREIRA BARRETO'],
      dtype=object)

In [ ]:
df_restricao = df_restricao.drop(['id_subsistema', 'nom_subsistema', 'id_estado', 'nom_estado', 'id_ons', 'ceg'], axis=1)

In [ ]:
df_restricao.head()

,nom_usina,din_instante,val_geracao,val_geracaolimitada,val_disponibilidade,val_geracaoreferencia,val_geracaoreferenciafinal,cod_razaorestricao,cod_origemrestricao,dsc_restricao
0,CONJ. BOA HORA,2025-01-01 00:00:00,0.0,NaN,30.017,0.0,NaN,NaN,NaN,NaN
1,CONJ. BOA HORA,2025-01-01 00:30:00,0.0,NaN,30.017,0.0,NaN,NaN,NaN,NaN
2,CONJ. BOA HORA,2025-01-01 01:00:00,0.0,NaN,30.082,0.0,NaN,NaN,NaN,NaN
3,CONJ. BOA HORA,2025-01-01 01:30:00,0.0,NaN,30.082,0.0,NaN,NaN,NaN,NaN
4,CONJ. BOA HORA,2025-01-01 02:00:00,0.0,NaN,30.082,0.0,NaN,NaN,NaN,NaN


In [ ]:
df_restricao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105120 entries, 0 to 105119
Data columns (total 10 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   nom_usina                   105120 non-null  object 
 1   din_instante                105120 non-null  object 
 2   val_geracao                 105120 non-null  float64
 3   val_geracaolimitada         19682 non-null   float64
 4   val_disponibilidade         105120 non-null  float64
 5   val_geracaoreferencia       105120 non-null  float64
 6   val_geracaoreferenciafinal  1111 non-null    float64
 7   cod_razaorestricao          19682 non-null   object 
 8   cod_origemrestricao         19682 non-null   object 
 9   dsc_restricao               7995 non-null    object 
dtypes: float64(5), object(5)
memory usage: 8.0+ MB


In [ ]:
df_restricao['din_instante'] = pd.to_datetime(df_restricao['din_instante'])

In [ ]:
df_restricao.isna().sum()

,0
nom_usina,0
din_instante,0
val_geracao,0
val_geracaolimitada,85438
val_disponibilidade,0
val_geracaoreferencia,0
val_geracaoreferenciafinal,104009
cod_razaorestricao,85438
cod_origemrestricao,85438
dsc_restricao,97125


Código da origem da restrição,
podendo ser:   
• LOC – Local.   
• SIS – Sistêmica  

Filtrar dados por horário 07:00 às 17:00

MWMed,Representa a potência média injetada naquele intervalo de 30 minutos.   


Para descobrir a energia física acumulada em Megawatts-hora ($\text{MWh}$), basta multiplicar o valor da coluna por $0,5 \text{ horas}$


`val_geracaolimitada` - Teto regulatório máximo de geração estipulado para aquela usina. A usina não pode passar desses valores que estão na coluna val_geracaolimitada.   


In [ ]:
df_restricao['time'] = df_restricao['din_instante'].dt.hour

In [ ]:
df_restricao.head(1)

,nom_usina,din_instante,val_geracao,val_geracaolimitada,val_disponibilidade,val_geracaoreferencia,val_geracaoreferenciafinal,cod_razaorestricao,cod_origemrestricao,dsc_restricao,time
0,CONJ. BOA HORA,2025-01-01,0.0,NaN,30.017,0.0,NaN,NaN,NaN,NaN,0


In [ ]:
df_restricao['time'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23], dtype=int32)

In [ ]:
df_restricao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105120 entries, 0 to 105119
Data columns (total 11 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   nom_usina                   105120 non-null  object        
 1   din_instante                105120 non-null  datetime64[ns]
 2   val_geracao                 105120 non-null  float64       
 3   val_geracaolimitada         19682 non-null   float64       
 4   val_disponibilidade         105120 non-null  float64       
 5   val_geracaoreferencia       105120 non-null  float64       
 6   val_geracaoreferenciafinal  1111 non-null    float64       
 7   cod_razaorestricao          19682 non-null   object        
 8   cod_origemrestricao         19682 non-null   object        
 9   dsc_restricao               7995 non-null    object        
 10  time                        105120 non-null  int32         
dtypes: datetime64[ns](1), float64(5), int32

In [ ]:
df_restricao = df_restricao[df_restricao['time'].between(7, 17)]

In [ ]:
df_restricao.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48180 entries, 14 to 105107
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   nom_usina                   48180 non-null  object        
 1   din_instante                48180 non-null  datetime64[ns]
 2   val_geracao                 48180 non-null  float64       
 3   val_geracaolimitada         19203 non-null  float64       
 4   val_disponibilidade         48180 non-null  float64       
 5   val_geracaoreferencia       48180 non-null  float64       
 6   val_geracaoreferenciafinal  1079 non-null   float64       
 7   cod_razaorestricao          19203 non-null  object        
 8   cod_origemrestricao         19203 non-null  object        
 9   dsc_restricao               7690 non-null   object        
 10  time                        48180 non-null  int32         
dtypes: datetime64[ns](1), float64(5), int32(1), object(4)
mem

In [ ]:
df_restricao['mes'] = (df_restricao['din_instante'].dt.month)

/tmp/ipykernel_905/1305505474.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_restricao['mes'] = (df_restricao['din_instante'].dt.month)


In [ ]:
df_restricao.head(2)

,nom_usina,din_instante,val_geracao,val_geracaolimitada,val_disponibilidade,val_geracaoreferencia,val_geracaoreferenciafinal,cod_razaorestricao,cod_origemrestricao,dsc_restricao,time,mes
14,CONJ. BOA HORA,2025-01-01 07:00:00,83.550,49.51,173.892,46.019,NaN,ENE,SIS,NaN,7,1
15,CONJ. BOA HORA,2025-01-01 07:30:00,52.969,53.00,117.685,56.804,NaN,ENE,SIS,NaN,7,1


In [ ]:
df_restricao.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48180 entries, 14 to 105107
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   nom_usina                   48180 non-null  object        
 1   din_instante                48180 non-null  datetime64[ns]
 2   val_geracao                 48180 non-null  float64       
 3   val_geracaolimitada         19203 non-null  float64       
 4   val_disponibilidade         48180 non-null  float64       
 5   val_geracaoreferencia       48180 non-null  float64       
 6   val_geracaoreferenciafinal  1079 non-null   float64       
 7   cod_razaorestricao          19203 non-null  object        
 8   cod_origemrestricao         19203 non-null  object        
 9   dsc_restricao               7690 non-null   object        
 10  time                        48180 non-null  int32         
 11  mes                         48180 non-null  int32        

Código da razão da restrição, podendo
ser:  
• REL – Razão de indisponibilidade
externa (elétrica).  
• CNF – Razão de atendimento a
requisitos de confiabilidade.   
• ENE – Razão energética   
• PAR - Restrição indicada no parecer
de acesso   

In [ ]:
usinas_validas = ['CONJ. CASTILHO', 'CONJ. DRACENA', 'CONJ. PEDRANÓPOLIS', 'CONJ. PEREIRA BARRETO']

df_restricao_filtrado = df_restricao[df_restricao['nom_usina'].isin(usinas_validas)]
df_restricao_filtrado.head()

,nom_usina,din_instante,val_geracao,val_geracaolimitada,val_disponibilidade,val_geracaoreferencia,val_geracaoreferenciafinal,cod_razaorestricao,cod_origemrestricao,dsc_restricao,time,mes
1502,CONJ. CASTILHO,2025-01-01 07:00:00,74.063,80.06,180.389,104.119,NaN,ENE,SIS,NaN,7,1
1503,CONJ. CASTILHO,2025-01-01 07:30:00,54.557,56.00,106.824,134.128,NaN,ENE,SIS,NaN,7,1
1504,CONJ. CASTILHO,2025-01-01 08:00:00,55.232,56.00,59.082,172.920,NaN,ENE,SIS,NaN,8,1
1505,CONJ. CASTILHO,2025-01-01 08:30:00,55.493,56.00,44.157,182.223,NaN,ENE,SIS,NaN,8,1
1506,CONJ. CASTILHO,2025-01-01 09:00:00,55.709,56.00,29.371,196.868,NaN,ENE,SIS,NaN,9,1


Entre ás 08:30-09:00 foi determinado que a usina não poderia passar de 56 MWmed (val_geracaolimitada). A operação da usina naquele intervalo val_gercao, não ultrapassa esse valor de 56, fica bem próximo do limite.    

Se não houvesse limites naquele intervalo a usina teria produzido val_geracaoreferencia 182-196MWmed. Essa difença foi perdida, geração perdida:


$$
\text{Geração Perdida} = \text{val_geracaoreferencia} - \text{val_geracaolimitada}
$$




**val_geracaoreferenciafinal** - Usada apenas para compensação financeira. A regulação brasileira estabelece que as usinas só têm direito a receber compensação financeira se o corte for do tipo REL (Razão Elétrica - falha em linhas de transmissão externas). ENE (Razão Energética - excesso de oferta geral de energia no país). Como cortes energéticos não dão direito a ressarcimento financeiro pela CCEE, o ONS deixa essa coluna preenchida como NaN.


Portanto, para a analise na qualidade dos dados, vou descartar essa coluna.







quantificar a perda causada pelo constrained-off:

$$\text{Energia Perdida (MWh)} = (\text{val_geracaoreferencia} - \text{val_geracao}) \times 0,5$$

In [ ]:
df_restricao_filtrado.isna().sum()

,0
nom_usina,0
din_instante,0
val_geracao,0
val_geracaolimitada,19526
val_disponibilidade,0
val_geracaoreferencia,0
val_geracaoreferenciafinal,31758
cod_razaorestricao,19526
cod_origemrestricao,19526
dsc_restricao,27203


In [ ]:
df_restricao_filtrado.groupby('nom_usina').size()

,0
nom_usina,
CONJ. CASTILHO,8030
CONJ. DRACENA,8030
CONJ. PEDRANÓPOLIS,8030
CONJ. PEREIRA BARRETO,8030


In [ ]:
df_restricao_filtrado[df_restricao_filtrado['val_geracaolimitada'].notna()].groupby('nom_usina').size()

,0
nom_usina,
CONJ. CASTILHO,3138
CONJ. DRACENA,3111
CONJ. PEDRANÓPOLIS,3105
CONJ. PEREIRA BARRETO,3240


In [ ]:
df_restricao_filtrado['val_geracaoreferenciafinal', 'cod_razaorestricao', 'cod_origemrestricao', 'dsc_restricao'].drop

array([    nan,  85.678,  85.297,   0.   ,  40.102,  22.05 ,  16.125,
        13.418,   4.296,  29.365,   1.752,   1.846,   3.287,   7.35 ,
        65.546,   8.412,   5.051,   4.9  , 159.242, 168.723,  68.434,
       111.05 ,  76.718,  63.831,  31.503,   5.047,   8.783,  49.441,
        16.685,  18.354,  24.554,  33.75 ,  34.806,  37.868,  28.033,
        31.527,  16.541,  24.818,   1.586,   8.354,  15.267,  18.204,
        23.342,  19.309,  11.572,   6.75 ,  29.78 ,  54.544,  75.192,
        80.782,  83.015,  83.57 ,  84.076,  84.31 ,  83.715,  83.556,
        83.187,  82.483,  82.31 ,  80.945,  80.826,  80.724,  79.815,
        79.267,  77.896,  75.367,  68.073,  47.898,  45.324,  67.067,
        77.747,  81.907,  82.834,  82.608,  82.851,  16.849,  24.274,
        37.558,  74.857,  80.264,  82.354,  77.557,  78.656,  63.384,
       142.39 , 186.548, 181.451, 166.448, 120.761, 168.084, 226.709,
       274.315, 272.994, 273.718, 274.367, 272.991, 272.215, 270.616,
       263.461, 269.

### Tabela Comparativa

In [ ]:
df_restricao_filtrado['perda_mwh'] = (df_restricao_filtrado['val_geracaoreferencia'] - df_restricao_filtrado['val_geracao']) * 0.5

# Calcula a perda percentual daquele instante de tempo
df_restricao_filtrado['perda_pct'] = ((df_restricao_filtrado['val_geracaoreferencia'] - df_restricao_filtrado['val_geracao']) / df_restricao_filtrado['val_geracaoreferencia']) * 100

tabela_qualidade = df_restricao_filtrado.groupby('nom_usina').agg(
    N_Eventos_Restricao=('cod_razaorestricao', 'count'),
    Energia_Limitada_MWh=('perda_mwh', 'sum'),
    Perda_Media_Pct=('perda_pct', 'mean'),
    Principal_Motivo=('cod_razaorestricao', lambda x: x.mode()[0] if not x.mode().empty else 'N/A')
).reset_index()

tabela_qualidade['Horas_Restritas'] = tabela_qualidade['N_Eventos_Restricao'] * 0.5

tabela_qualidade = tabela_qualidade[['nom_usina', 'N_Eventos_Restricao', 'Horas_Restritas', 'Energia_Limitada_MWh', 'Perda_Media_Pct', 'Principal_Motivo']]

/tmp/ipykernel_905/818335260.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_restricao_filtrado['perda_mwh'] = (df_restricao_filtrado['val_geracaoreferencia'] - df_restricao_filtrado['val_geracao']) * 0.5
/tmp/ipykernel_905/818335260.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_restricao_filtrado['perda_pct'] = ((df_restricao_filtrado['val_geracaoreferencia'] - df_restricao_filtrado['val_geracao']) / df_restricao_filtrado['val_geracaoreferencia']) * 100


### Tabela 1

In [ ]:
tabela_qualidade

,nom_usina,N_Eventos_Restricao,Horas_Restritas,Energia_Limitada_MWh,Perda_Media_Pct,Principal_Motivo
0,CONJ. CASTILHO,3138,1569.0,80467.7825,-inf,ENE
1,CONJ. DRACENA,3111,1555.5,3685.4875,-inf,ENE
2,CONJ. PEDRANÓPOLIS,3105,1552.5,27306.5730,-inf,ENE
3,CONJ. PEREIRA BARRETO,3240,1620.0,-141187.7055,-inf,ENE


In [ ]:
(df_restricao_filtrado['val_geracaoreferencia'] <=0).sum()

np.int64(750)

In [ ]:
filtro_ref_zero = df_restricao_filtrado['val_geracaoreferencia'] <= 0
colunas_analise = ['din_instante', 'nom_usina', 'val_geracao', 'val_geracaoreferencia']
df_restricao_filtrado[filtro_ref_zero][colunas_analise].head()

,din_instante,nom_usina,val_geracao,val_geracaoreferencia
1746,2025-01-06 09:00:00,CONJ. CASTILHO,193.158,0.0
1747,2025-01-06 09:30:00,CONJ. CASTILHO,198.577,0.0
1748,2025-01-06 10:00:00,CONJ. CASTILHO,200.683,0.0
1749,2025-01-06 10:30:00,CONJ. CASTILHO,186.373,0.0
1852,2025-01-08 14:00:00,CONJ. CASTILHO,198.576,0.0


In [ ]:
filtro_geracao_maior = df_restricao_filtrado['val_geracao'] > df_restricao_filtrado['val_geracaoreferencia']
df_restricao_filtrado[filtro_geracao_maior][colunas_analise].head()

,din_instante,nom_usina,val_geracao,val_geracaoreferencia
1517,2025-01-01 14:30:00,CONJ. CASTILHO,48.300,47.341
1518,2025-01-01 15:00:00,CONJ. CASTILHO,82.712,78.519
1519,2025-01-01 15:30:00,CONJ. CASTILHO,112.822,100.030
1550,2025-01-02 07:00:00,CONJ. CASTILHO,73.662,72.264
1551,2025-01-02 07:30:00,CONJ. CASTILHO,117.114,113.161


In [ ]:
total_linhas_maiores = (df_restricao_filtrado['val_geracao'] > df_restricao_filtrado['val_geracaoreferencia']).sum()
print(f"Quantidade de linhas onde a geração real superou a estimativa do ONS: {total_linhas_maiores}")

Quantidade de linhas onde a geração real superou a estimativa do ONS: 16327


### Tabela 2

In [ ]:
df_restricao_filtrado['perda_mw'] = df_restricao_filtrado['val_geracaoreferencia'] - df_restricao_filtrado['val_geracao']

df_restricao_filtrado['perda_mw'] = df_restricao_filtrado['perda_mw'].clip(lower=0) #Remover linhas qnd nao houve restrição para evitar valores negativos

df_restricao_filtrado['perda_mwh'] = df_restricao_filtrado['perda_mw'] * 0.5

df_restricao_filtrado['perda_pct'] = np.where(
    df_restricao_filtrado['val_geracaoreferencia'] > 0,
    (df_restricao_filtrado['perda_mw'] / df_restricao_filtrado['val_geracaoreferencia']) * 100,
    0.0
)

tabela_qualidade = df_restricao_filtrado.groupby('nom_usina').agg(
    N_Eventos_Restricao=('cod_razaorestricao', 'count'),
    Energia_Limitada_MWh=('perda_mwh', 'sum'),
    Perda_Media_Pct=('perda_pct', 'mean'),
    Principal_Motivo=('cod_razaorestricao', lambda x: x.mode()[0] if not x.mode().empty else 'N/A')
).reset_index()

tabela_qualidade['Horas_Restritas'] = tabela_qualidade['N_Eventos_Restricao'] * 0.5

tabela_qualidade = tabela_qualidade[['nom_usina', 'N_Eventos_Restricao', 'Horas_Restritas', 'Energia_Limitada_MWh', 'Perda_Media_Pct', 'Principal_Motivo']]

tabela_qualidade = tabela_qualidade.round({'Energia_Limitada_MWh': 2, 'Perda_Media_Pct': 2})

tabela_qualidade

/tmp/ipykernel_905/1236423783.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_restricao_filtrado['perda_mw'] = df_restricao_filtrado['val_geracaoreferencia'] - df_restricao_filtrado['val_geracao']
/tmp/ipykernel_905/1236423783.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_restricao_filtrado['perda_mw'] = df_restricao_filtrado['perda_mw'].clip(lower=0)
/tmp/ipykernel_905/1236423783.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

,nom_usina,N_Eventos_Restricao,Horas_Restritas,Energia_Limitada_MWh,Perda_Media_Pct,Principal_Motivo
0,CONJ. CASTILHO,3138,1569.0,96127.00,17.32,ENE
1,CONJ. DRACENA,3111,1555.5,25604.74,17.22,ENE
2,CONJ. PEDRANÓPOLIS,3105,1552.5,35213.42,13.79,ENE
3,CONJ. PEREIRA BARRETO,3240,1620.0,83672.53,11.96,ENE


In [ ]:
tabela_qualidade

,nom_usina,N_Eventos_Restricao,Horas_Restritas,Energia_Limitada_MWh,Perda_Media_Pct,Principal_Motivo
0,CONJ. CASTILHO,3138,1569.0,96127.00,17.32,ENE
1,CONJ. DRACENA,3111,1555.5,25604.74,17.22,ENE
2,CONJ. PEDRANÓPOLIS,3105,1552.5,35213.42,13.79,ENE
3,CONJ. PEREIRA BARRETO,3240,1620.0,83672.53,11.96,ENE


*   O número de eventos de restrição é praticamente idêntico para as quatro usinas.
*   As usinas solares em São Paulo passaram cerca de 40% do tempo sofrendo algum tipo de corte pelo ONS
*   O corte não é por falha na subestação delas, mas sim porque o Brasil está gerando muita energia (provavelmente excesso de hidráulica ou eólica de outras regiões) e, para equilibrar o sistema nacional, o ONS manda o Norte/Oeste paulista cortar geração.
*   Embora Pereira Barreto seja um complexo gigantesco (457 MW), o Conjunto Castilho perdeu mais energia em termos absolutos (96.127 MWh vs. 83.672 MWh). Isso acontece porque a severidade média do corte em Castilho foi maior (17,32% de perda média por evento contra 11,96% em Pereira Barreto).



As principais conclusões são:
*   frequência semelhante de eventos entre todas as usinas;
*   aproximadamente 40% do período analisado apresentou algum tipo de restrição;
*   predominância da razão ENE;
*   diferenças na severidade das perdas, com Castilho apresentando maior perda percentual média e maior energia limitada absoluta